# Monitor and retrain

Read [lesson 05](../../docs/05-monitoring-and-retraining.md). This notebook uses a
temporary registry, synthetic traffic and the real API through a test client.
It never writes to the command-line production registry.


In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from fastapi.testclient import TestClient
from iris_mlops.data import FEATURES, prepare_data, sha256
from iris_mlops.workflow import train_run, promote, export_bundle, registry
from iris_mlops.serve import create_app
from iris_mlops.monitor import read_events, summarize
from iris_mlops.retrain import retrain

workspace = TemporaryDirectory()
work = Path(workspace.name)
data_dir, state_dir = work / "data", work / "state"
frames = prepare_data(data_dir)
version = train_run(data_dir, root=state_dir, run_id="monitor-baseline")
promote(version, root=state_dir)
bundle = export_bundle(version, work / "release", root=state_dir)
log = work / "predictions.jsonl"


## Send normal and deliberately corrupted traffic

These are replays of training examples, not a generalization benchmark. The
second request adds 5 cm to petal length but keeps the original labels to model
an upstream measurement incident. Such corrupted data should be repaired,
not automatically appended to training.


In [ ]:
feedback = []
with TestClient(create_app(bundle, log_path=log)) as client:
    for shift in [0, 5]:
        batch = frames["train"].copy()
        batch["petal_length"] += shift
        response = client.post("/predict", json={"instances": batch[FEATURES].to_dict("records")})
        response.raise_for_status()
        result = response.json()
        feedback.extend({"request_id": result["request_id"], "row_index": i, "label": label}
                        for i, label in enumerate(batch.label))
events = read_events(log)
print(f"Logged {len(events)} requests and {len(feedback)} individual labels")


## First, analyze without labels

A feature shift can be observed immediately. Accuracy cannot be measured from
predictions alone; the report must leave it unknown.


In [ ]:
unlabeled = summarize(bundle, events)
assert unlabeled["accuracy"] is None
display(unlabeled)
pd.Series(unlabeled["drift"], name="Mean shift in training std units").plot.bar(rot=20)
plt.axhline(1, color="black", linestyle="--")
plt.tight_layout()
plt.show()


## Then join independently supplied labels

Compare the normal and corrupted windows separately. In a real system, also
inspect the percentage and age of predictions that have reliable labels.


In [ ]:
normal = summarize(bundle, events[:1], feedback)
corrupted = summarize(bundle, events[1:], feedback)
display(pd.DataFrame([normal, corrupted], index=["normal replay", "corrupted replay"])
        [["samples", "labeled_samples", "accuracy", "latency_p95_ms", "alerts"]])


## Retrain from reviewed new observations

The supplied six-row CSV is synthetic teaching data with separate IDs. The
retraining helper preserves validation/test bytes and never promotes itself.


In [ ]:
from iris_mlops.paths import PROJECT_DIR

result = retrain(PROJECT_DIR / "examples/new-labels.csv", work / "data-v2", data_dir, state_dir)
display(result)
for split in ["validation", "test"]:
    assert sha256(data_dir / f"{split}.csv") == sha256(work / "data-v2" / f"{split}.csv")
assert registry(state_dir)["production"] == version
print("Holdouts unchanged; production still serves the reviewed baseline")


## Decide the next action

- Should the corrupted input incident cause a model rollback or an upstream repair?
- What evidence makes a new label trustworthy?
- When is there enough labeled traffic to act on an accuracy alert?
- What would you review before promoting this candidate?

Continue with the same evaluation, export, staging and smoke steps from the
local lifecycle notebook. Do not use alerts as automatic production approval.
